# Notebook couche Gold

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, avg

# 1. Spark Session
spark = SparkSession.builder \
    .appName("water-quality-gold") \
    .getOrCreate()

# 2. Chargement de la couche Silver

silver_path = "./data/silver/water_quality_clean/"

print(f"Loading Silver layer from: {silver_path}")

df = spark.read.parquet(silver_path)

print("Silver schema:")
df.printSchema()

print(f"Total rows loaded: {df.count()}")

# Aperçu des données
df.show(5, truncate=False)

### Table: Conformité par commune

In [ ]:
print("\nCreating: conformite_commune")

df_commune = (
    df.groupBy("commune")
    .agg(
        count("*").alias("total_tests"),

        count(
            when(col("categorie") == "Conforme", True)
        ).alias("nb_conformes"),

        count(
            when(col("categorie") != "Conforme", True)
        ).alias("nb_non_conformes")
    )
    .withColumn(
        "taux_conformite",
        col("nb_conformes") / col("total_tests")
    )
)

# Aperçu
print("\nPreview: conformite_commune")
df_commune.orderBy(col("taux_conformite").asc()).show(10, truncate=False)

# Sauvegarde
output_path = "./data/gold/conformite_commune/"

df_commune.write.mode("overwrite").parquet(output_path)

print(f"Saved: {output_path}")

### Table: Evolution temporelle

In [ ]:
print("\nCreating: evolution_temporelle")

df_time = (
    df.groupBy("annee", "parametre_nom")
    .agg(
        avg("resultat_valeur_float").alias("moyenne_resultat"),
        count("*").alias("nb_mesures")
    )
)

# Aperçu
print("\nPreview: evolution_temporelle")
df_time.show(10, truncate=False)

# Sauvegarde
output_path = "./data/gold/evolution_temporelle/"

df_time.write.mode("overwrite").parquet(output_path)

print(f"Saved: {output_path}")

### Table: Top 10 communes les moins conformes

In [ ]:
print("\nCreating: top_10_pires_communes")

df_worst = (
    df_commune
    .orderBy(col("taux_conformite").asc())
    .limit(10)
)

# Aperçu
print("\nPreview: top_10_pires_communes")
df_worst.show(truncate=False)

# Sauvegarde
output_path = "./data/gold/top_10_pires_communes/"

df_worst.write.mode("overwrite").parquet(output_path)

print(f"Saved: {output_path}")


### Table: Non conformité par paramètre

In [ ]:
print("\nCreating: non_conformites_parametre")

df_non_conform = (
    df.filter(col("categorie") != "Conforme")
    .groupBy("parametre_nom")
    .agg(
        count("*").alias("nb_non_conformes")
    )
    .orderBy(col("nb_non_conformes").desc())
)

# Aperçu
print("\nPreview: non_conformites_parametre")
df_non_conform.show(10, truncate=False)

# Sauvegarde
output_path = "./data/gold/non_conformites_parametre/"

df_non_conform.write.mode("overwrite").parquet(output_path)

print(f"Saved: {output_path}")

In [ ]:
print("GOLD LAYER SUCCESSFULLY GENERATED")

print("\nGenerated datasets:")
print("- conformite_commune")
print("- evolution_temporelle")
print("- top_10_pires_communes")
print("- non_conformites_parametre")